# End to end 2 — Design: from "we should test treatment a" to a costed, powered, chosen design

**The question.** A fitted response surface says treatment `a` works. Someone proposes an
experiment to confirm it. Should it run, at what size, with which estimator, and is it worth
what it costs?

The route through `axiom.design`:

1. **What effect are we testing?** The prior comes from the fitted surface (`surface.fit` →
   `estimands.realize`), and `anchor_effect` turns the posterior into the effect a test should
   be *powered for* — the one the model doubts, not the one it already believes.
2. **How big?** `power` / `mde` / `sample_size` from one normal-model formula.
3. **Which estimator?** The `METHODS` registry, A/A-calibrated on a panel like ours, and
   `simulated_power` at the anchored effect.
4. **What is the information worth?** `eig_gaussian` for nats, `evoi_gaussian` for money on a
   stated decision, `opportunity_cost` / `experiment_value` for the net.
5. **Which of three concrete designs?** `evaluate_candidate`, `pareto_front`, and `perturb`
   for how robust the winner is.
6. **What dose schedule identifies carryover?** `Schedule` patterns scored by `contrast_score`
   and confirmed by `fisher_information`.

Crossed subpackages: `sim` → `surface` → `estimands` → `design`.

In [ ]:
import numpy as np

from axiom.core import Population, TimeWindow, is_failure
from axiom.design import (
    METHODS, AnchoredEffect, DecisionSpec, DesignCandidate, EconomicInputs, FisherInformation, MDE,
    SampleSize, SimulatedPower, SimulationSpec, ValuePerOutcome, anchor_draws, calibrate_registry,
    constant, contrast_score, difference_in_differences_se, difference_se, eig_gaussian, evaluate_candidate, evoi_gaussian,
    experiment_value, fisher_information, information_value_of, mde, opportunity_cost, pareto_front,
    perturb, power, power_from_se, pulse, random_switchback, sample_size, simulated_power,
)
from axiom.estimands import Level, RealizedDraws, realize, standard_estimands
from axiom.identify import CausalGraph, identify
from axiom.sim import DosePlan, arms_world, surface_world
from axiom.surface import GeometricCarryover, HillKernel, fit

SEED = 0

## 1. The belief we start from

A dose-finding study already exists: 40 arms of treatment `a` spread over `0–100` on a Hill
surface. Fit it with
the Laplace backend and realize the estimand the decision actually cares about — the
*contrast* between dosing at 60 and not dosing. `keep_draws=True` keeps the posterior draws
of that contrast, which is the prior the experiment will update.

In [ ]:
prior_world = arms_world(
    n_units=40, treatments=("a",), kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
    doses={"a": np.linspace(0.0, 100.0, 40)},
    truth={"beta_a": 6.0, "k_a": 50.0, "s_a": 2.0, "alpha": 1.0}, noise_sd=1.5, seed=SEED,
)
res = fit(prior_world.spec, prior_world.panel, backend="laplace", draws=2000, seed=SEED)
print("converged:", res.converged, "| outcome sd (posterior mean of sigma):", round(res.posterior.summary("sigma").mean, 3))

registry = standard_estimands(
    treatment=prior_world.spec.treatments[0], outcome=prior_world.spec.outcome, population=Population(name="all"),
    window=TimeWindow(start=0, stop=1), level=Level(unit="individual"), dose=60.0, reference_dose=0.0,
)
verdict = identify(CausalGraph.from_edges("a -> y"), "a", "y").verdict
rd = realize(registry.get("contrast_at_dose"), res, verdict=verdict, keep_draws=True, mass=0.9)
assert isinstance(rd, RealizedDraws)
contrast_draws = np.asarray(rd.draws).ravel()
prior_mean, prior_sd = float(contrast_draws.mean()), float(contrast_draws.std())
truth_contrast = float(prior_world.forward({"a": 60.0}).mean() - prior_world.forward({"a": 0.0}).mean())
print(f"prior on the contrast: {prior_mean:.3f} ± {prior_sd:.3f}  {rd.result.summary.interval}  (truth {truth_contrast:.3f})")

## 2. Anchor the effect size

The conventional MDE for this outcome is `2.0`. If the posterior already puts nearly all its
mass above that, an experiment powered for `2.0` confirms a belief rather than testing it.
`anchor_draws` reports `P(effect > MDE)` and the `1 − credence` quantile — the effect the
model doubts at 90 % credence — and that is the size the test is powered for.

In [ ]:
MDE_CONVENTIONAL = 2.0
ae = anchor_draws(contrast_draws, "contrast_at_dose", mde=MDE_CONVENTIONAL, credence=0.9)
assert isinstance(ae, AnchoredEffect)
print(f"P(contrast > {MDE_CONVENTIONAL}) = {ae.probability_exceeds_mde:.3f}; already believed: {ae.already_believed}")
print(f"anchored effect (10% quantile) = {ae.anchored_effect:.3f}")
effect = ae.anchored_effect if ae.already_believed else MDE_CONVENTIONAL
print("effect the experiment is powered for:", round(effect, 3))

## 3. Power, MDE, sample size

One formula. The outcome sd comes from the fitted surface's `sigma`. `sample_size` is the
exact inversion of `power`, so the achieved power at the returned `n` is the target, not an
approximation; `mde` answers the converse question for a size someone has already proposed.

In [ ]:
sd = float(res.posterior.summary("sigma").mean)
ss = sample_size(effect=effect, sd=sd, power=0.8, alpha=0.05)
assert isinstance(ss, SampleSize)
print(f"n for effect {effect:.2f} at sd {sd:.2f}: {ss.n} ({ss.n_treated} treated / {ss.n_control} control), power {ss.power:.3f}")
proposed_n = 40
m: MDE = mde(proposed_n, sd=sd, power=0.8)
print(f"the proposed n={proposed_n} detects {m.effect:.3f} at 80% power; power for {effect:.2f} is {power(proposed_n, effect, sd).power:.3f}")
se_experiment = difference_se(ss.n, sd=sd, allocation=ss.n_treated / ss.n)
print(f"experiment se at n={ss.n}: {se_experiment:.3f}")

## 4. Choose the estimator

The experiment will be a panel: units observed before and after, some held out. Six
estimators share one `PanelArrays` contract in `METHODS`. Before trusting any of them on a
panel of this shape, calibrate them under the null (`calibrate_registry` returns a *new*
registry with statuses set from the A/A false-positive count, judged against the exact
binomial region; a method whose false-positive count leaves the region is marked
`experimental` in the returned registry and is not a candidate below), then check the
realized power at the anchored effect with `simulated_power`. The single-period formula from step 3 is conservative for a panel: the
DiD contrast averages noise over the pre and post windows, and
`difference_in_differences_se` gives its exact standard error under this DGP.

In [ ]:
for name, spec in METHODS.items():
    print(f"{name:28s} pre={spec.requires_pre_period!s:5} controls={spec.requires_controls!s:5} {spec.assumption_names()}")

panel_spec = SimulationSpec(
    n_units=ss.n, n_periods=12, n_pre=6, n_treated=ss.n_treated, noise_sd=sd, n_simulations=40, seed=SEED,
)
calibrated, results = calibrate_registry(panel_spec, alpha=0.05)
print()
for r in results:
    print(f"{r.method:28s} design={r.design:10s} fp={r.false_positive_count}/{r.n_evaluated} "
          f"region=[{r.region.lower}, {r.region.upper}] passed={r.passed} -> {calibrated[r.method].status}")

In [ ]:
ab = panel_spec.model_copy(update={"effect": effect})
single_period = power(ss.n, effect, sd).power
se_did = difference_in_differences_se(ab)
predicted = power_from_se(effect, se_did).power
chosen_method = "difference_in_differences"
sp: SimulatedPower = simulated_power(chosen_method, ab, predicted_power=predicted)
print(f"single-period formula power {single_period:.3f}; exact DiD se over {ab.n_pre}+{ab.n_post} periods {se_did:.3f} -> predicted {predicted:.3f}")
print(f"{chosen_method}: realized {sp.power:.3f} ({sp.rejections}/{sp.n_evaluated}), "
      f"bias {sp.bias:+.3f}, coverage {sp.coverage:.2f}, within prediction: {sp.within_prediction}")

## 5. What is the information worth?

`eig_gaussian` gives the expected information gain in nats for a Gaussian prior measured at
the experiment's standard error. To turn it into money, state the decision: *scale up*
treatment `a` across the whole program if the contrast exceeds `3.0` per unit. An outcome
unit is worth 400 in the numeraire and the program covers 500 units, so each unit of contrast
the decision is right about is worth `200 000` — the experiment withholds dose from a handful
of units to inform a decision about all of them. `evoi_gaussian` reports EVPI (a clairvoyant's
gain over acting on the prior mean) and EVSI (what *this* experiment's answer is worth).

In [ ]:
print(f"EIG at se {se_experiment:.3f}: {eig_gaussian(prior_sd, se_experiment):.3f} nats")
VALUE_PER_OUTCOME, PROGRAM_UNITS = 400.0, 500
decision = DecisionSpec(name="scale_up", threshold=3.0, value_per_outcome_unit=VALUE_PER_OUTCOME * PROGRAM_UNITS, numeraire="USD")
ev = evoi_gaussian(decision, prior_mean, prior_sd, se_experiment)
print(f"EVPI {ev.evpi:.2f} {ev.numeraire} | EVSI {ev.evsi:.2f} {ev.numeraire} | preposterior sd {ev.preposterior_sd:.3f}")

## 6. What it costs

Holding units out withholds dose, and withholding dose forgoes outcome. `opportunity_cost`
is signed — a treatment the prior thinks is net-negative has a *negative* cost of withholding
— and takes the marginal value ratio as draws so its uncertainty is carried. `ValuePerOutcome`
records where the conversion came from as a ledger line. `experiment_value` nets EVSI against
the opportunity and fixed costs.

In [ ]:
vpo = ValuePerOutcome(value=VALUE_PER_OUTCOME, outcome_unit="unit", numeraire="USD", source="stated by the decision owner, 2026")
print(vpo.ledger_line().statement)
ratio_draws = contrast_draws / 60.0  # outcome units per dose unit at the tested dose
oc = opportunity_cost(
    ss.n_control / ss.n, panel_spec.n_post, dose_per_period=60.0, marginal_value_ratio=ratio_draws,
    value_per_outcome=vpo, discount_rate=0.01, dose_unit="USD", dose_cost_per_unit=1.0,
)
print(f"dose withheld {oc.dose_withheld:.0f}; outcome forgone {oc.outcome_forgone:.1f}; opportunity cost {oc.value:.2f} {oc.numeraire}")
info = information_value_of(decision, prior_mean=prior_mean, prior_sd=prior_sd, experiment_se=se_experiment)
value = experiment_value(info, oc, fixed_cost=500.0)
print(f"information {value.information_value:.2f} − opportunity {value.opportunity_cost:.2f} − fixed {value.fixed_cost:.2f} = net {value.net:.2f} {value.numeraire}")

## 7. Three concrete designs

Each `DesignCandidate` is one way of running it: a calibrated method, a size, a horizon, a
holdout share, the standard error it would achieve, and a cost. The three here are the
minimal powered holdout from step 3, the larger holdout someone originally proposed, and a
switchback of the powered size (a smaller standard error per unit, since every unit serves as
its own control, but more dose withheld over time). `evaluate_candidate` scores
all three on the same decision and prior; `pareto_front` keeps the ones not dominated on net
value, cost (minimized), and EIG.

In [ ]:
economics = EconomicInputs(
    value_per_outcome=vpo, dose_per_period=60.0, discount_rate=0.01, dose_unit="USD", dose_cost_per_unit=1.0,
    marginal_value_ratio=float(ratio_draws.mean()),
)
candidates = [
    DesignCandidate(name="powered_holdout", method="difference_in_differences", n_units=ss.n, n_periods=12,
                    holdout_fraction=ss.n_control / ss.n, experiment_se=se_experiment, cost=300.0, cooldown_periods=2),
    DesignCandidate(name="proposed_holdout", method="difference_in_differences", n_units=proposed_n, n_periods=12,
                    holdout_fraction=0.5, experiment_se=difference_se(proposed_n, sd=sd), cost=550.0, cooldown_periods=2),
    DesignCandidate(name="switchback", method="switchback", n_units=ss.n, n_periods=16, holdout_fraction=0.5,
                    experiment_se=0.8 * se_experiment, cost=420.0, cooldown_periods=1),
]
scores = [evaluate_candidate(c, decision, prior_mean=prior_mean, prior_sd=prior_sd, economics=economics) for c in candidates]
for s in scores:
    print(f"{s.name:16s} eig={s.eig:.3f} evsi={s.evsi:8.2f} oc={s.opportunity_cost:8.2f} cost={s.cost:6.1f} net={s.net_value:8.2f} power={s.power:.3f}")
front = pareto_front(scores, objectives=("net_value", "-cost", "eig"))
print("Pareto front:", [s.name for s in front])
winner = max(scores, key=lambda s: s.net_value)
print("highest net value:", winner.name)

## 8. How robust is the winner?

`perturb` re-scores every candidate along a grid of one input and records the winner at each
point. A *tipping point* is a pair of adjacent grid values across which the winner changes.
Two inputs matter most: the value of an outcome unit (a business number someone stated) and
the experiment's standard error (a statistical number the design promised).

In [ ]:
for parameter, grid in (("value_per_outcome", (0.25, 0.5, 1.0, 2.0, 4.0)), ("experiment_se", (0.5, 0.75, 1.0, 1.5, 2.0))):
    table = perturb(candidates, decision, prior_mean, prior_sd, economics, parameter, grid=grid)
    print(f"{parameter} ({table.mode}): winners {table.winners}")
    print(f"   base winner {table.base_winner} | stable {table.stable} | tipping points {table.tipping_points}")

## 9. The schedule that identifies carryover

If `a` has carryover, the holdout's *level* contrast identifies the effect size but a dose
held constant passes through the carryover unchanged — the data carry no information on the
decay rate. Identification of `lam_a` comes from temporal contrast. `contrast_score` is the
scale-free heuristic (0 for a flat schedule); `fisher_information` on the surface is the
model-based verdict, evaluated at plausible parameters.

In [ ]:
T, NOISE_SD = 12, 3.0
carry_world = surface_world(
    n_units=2, n_periods=T, treatments=("a",), kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
    carryover=GeometricCarryover(max_lag=4), doses=DosePlan(scale=50.0), intercept="shared",
    truth={"beta_a": 6.0, "alpha": 1.0, "k_a": 50.0, "s_a": 2.0, "lam_a": 0.5}, noise_sd=NOISE_SD, seed=SEED,
)
schedules = [constant(T, 60.0), pulse(T, 100.0, 20.0, on=2, off=2), random_switchback(T, 100.0, 20.0, seed=SEED)]
for s in schedules:
    data = dict(carry_world.data)
    data["a"] = s.as_grid(2)
    fi = fisher_information(carry_world.surface, data, carry_world.theta, NOISE_SD, method="finite")
    assert isinstance(fi, FisherInformation)
    i = fi.index("lam_a")
    print(f"{s.pattern:18s} mean dose {s.mean:5.1f}  contrast={contrast_score(s):.3f}  info[lam_a]={fi.as_array()[i, i]:9.3f}  singular={fi.singular}")

## What this notebook decided

- The experiment is powered for the **anchored** contrast, not the conventional MDE, because
  the fitted surface already believed the conventional one.
- `sample_size` gave the *minimal* `n`; the A/A calibration showed which registered estimators
  are honest on a panel of that shape, and `simulated_power` confirmed the power of the chosen
  one. Power is a floor, not the criterion.
- The information is worth a stated amount in the numeraire on the stated decision; the
  opportunity cost of the holdout is priced with its uncertainty and a ledger line on the
  conversion.
- Of three concrete designs, net value — EVSI less the priced holdout and the fixed cost —
  picks the winner; the Pareto front shows what is traded away, and `perturb` says how far the
  value of an outcome unit and the promised standard error can move before the choice changes.
- If carryover matters, the dose schedule must alternate: a flat schedule has zero contrast
  and no information on the decay, whatever its size.